# My first electricity demand forecasting project

**Question:** Can we predict daily electricity demand using calendar patterns and temperature?

We will make one forecast for January 2025, pretending we are at the end of December 2024 and that December 31 demand is available. This is a historical learning exercise, not a live forecast or a fresh independent test. We already studied 2025 in the earlier project.

We will use Python for analysis, SQLite for two simple checks, and linear regression for prediction. No parameter search, bootstrap or uncertainty intervals are needed for this first version.

Run one code cell at a time. Read the explanation, run the cell, and inspect its output before moving on.

## 1. Load the data

`pandas` helps us work with tables. `read_csv()` opens the prepared dataset. A DataFrame is simply Python's table of rows and columns.

The data was already downloaded from EIA and Open-Meteo. We use the preserved file rather than learning API access at the same time. EIA demand is daily energy in **MWh**. Temperature comes from four Texas cities.

In [1]:
# Import the tools we will use and find the project folder.
from pathlib import Path
import os
import sqlite3
import pandas as pd
import numpy as np

# Find the repository when launched from its root or notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'data/processed/daily_demand_weather.csv').exists(), 'Start Jupyter from the repository root.'
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.mplconfig'))
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Load the existing file and convert date text into real dates.
data_file = ROOT / 'data/processed/daily_demand_weather.csv'
print('Loading:', data_file)
df = pd.read_csv(data_file)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
print('Rows and columns:', df.shape)
display(df.head())


Loading: <project-root>/data/processed/daily_demand_weather.csv
Rows and columns: (2557, 18)


,date,demand_mwh,houston_temperature_2m_mean,houston_temperature_2m_max,houston_temperature_2m_min,houston_relative_humidity_2m_mean,dallas_temperature_2m_mean,dallas_temperature_2m_max,dallas_temperature_2m_min,dallas_relative_humidity_2m_mean,austin_temperature_2m_mean,austin_temperature_2m_max,austin_temperature_2m_min,austin_relative_humidity_2m_mean,san_antonio_temperature_2m_mean,san_antonio_temperature_2m_max,san_antonio_temperature_2m_min,san_antonio_relative_humidity_2m_mean
0,2019-01-01,985428.0,11.8,17.3,5.5,80,4.4,7.4,2.7,79,6.3,8.6,4.4,80,7.1,10.2,3.9,78
1,2019-01-02,1137156.0,10.9,13.5,9.4,89,2.0,2.7,1.1,89,3.8,5.1,3.1,93,4.4,5.8,3.6,95
2,2019-01-03,1100758.0,9.5,13.1,5.1,91,3.6,4.4,2.9,93,5.6,11.2,2.3,84,6.8,13.4,3.3,80
3,2019-01-04,1013134.0,7.9,15.7,2.4,78,7.3,14.9,2.8,71,8.3,18.8,1.3,65,9.9,20.9,2.9,57
4,2019-01-05,903604.0,11.3,21.9,3.6,77,10.0,21.3,1.7,72,10.8,23.2,1.2,68,10.9,22.4,1.7,65


## 2. Check the data with SQL

Before modeling, check that the dates and values make sense. SQL is a language for asking questions about tables.

We copy the data into a temporary SQLite database. `COUNT(*)` counts rows; `COUNT(demand_mwh)` counts rows with known demand. The second query shows the missing demand date. We do not replace unknown demand with zero.

In [2]:
# Store a copy in a temporary database; the source CSV remains unchanged.
connection = sqlite3.connect(':memory:')
df.to_sql('electricity', connection, index=False, if_exists='replace')

# Compare the number of calendar days with the number of known demand values.
quality = pd.read_sql_query("""
SELECT COUNT(*) AS calendar_days,
       COUNT(demand_mwh) AS days_with_demand
FROM electricity
""", connection)
display(quality)

# Find the dates where demand is unknown.
missing = pd.read_sql_query("""
SELECT date FROM electricity WHERE demand_mwh IS NULL
""", connection)
display(missing)
connection.close()

# A duplicate date would confuse the daily sequence, so stop if we find one.
assert not df['date'].duplicated().any()


,calendar_days,days_with_demand
0,2557,2556


,date
0,2025-12-05 00:00:00


## 3. Prepare a small table

We only need a date, demand and temperature. To keep this exercise simple, average the four cities' mean temperatures. This is an approximation of regional weather.

Training data teaches the model. Test data checks its predictions. We train on 2019–2024 and predict the 31 days in January 2025. The known missing demand date is outside these periods.

In [3]:
# Select the four daily mean-temperature columns and calculate their average.
temperature_columns = [column for column in df.columns if column.endswith('temperature_2m_mean')]
assert len(temperature_columns) == 4
assert df[temperature_columns].notna().all().all()
df['temperature_c'] = df[temperature_columns].mean(axis=1)

# Keep only the columns needed for this first project.
data = df[['date', 'demand_mwh', 'temperature_c']].copy()
train = data[data['date'] < '2025-01-01'].copy()
test = data[(data['date'] >= '2025-01-01') & (data['date'] <= '2025-01-31')].copy()
assert train['demand_mwh'].notna().all()
assert len(test) == 31 and test['demand_mwh'].notna().all()
print('Training days:', len(train))
print('Test days:', len(test))
display(train.head())


Training days: 2192
Test days: 31


,date,demand_mwh,temperature_c
0,2019-01-01,985428.0,7.400
1,2019-01-02,1137156.0,5.275
2,2019-01-03,1100758.0,6.375
3,2019-01-04,1013134.0,8.350
4,2019-01-05,903604.0,10.750


## 4. Explore before predicting

Look at demand over time and how it relates to temperature. We use only the training data here.

The temperature relationship can curve: both very hot and very cold days may need more electricity. A chart shows an association; it does not prove that temperature alone caused the changes.

In [4]:
# Plot the history and temperature relationship using training data only.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train['date'], train['demand_mwh'] / 1_000_000, linewidth=0.7)
axes[0].set(title='Historical demand', xlabel='Date', ylabel='Million MWh per day')
axes[1].scatter(train['temperature_c'], train['demand_mwh'] / 1_000_000, s=6, alpha=0.3)
axes[1].set(title='Temperature and demand', xlabel='Temperature (°C)', ylabel='Million MWh per day')
plt.tight_layout()
plt.show()


/var/folders/51/k8dwm_2d6zx1zlrm0dq6jf9w0000gp/T/ipykernel_32679/1957645699.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Create model inputs

A **feature** is information the model uses to predict demand. Our features are temperature, temperature squared, month and weekday.

Squaring temperature lets the regression represent a curved relationship. `get_dummies()` gives each month/weekday its own indicator column rather than treating weekdays as a numeric ranking.

We do not know January's actual weather at the forecast date. For this beginner exercise, use the average January temperature from training data. This is a weather assumption, not a real weather forecast. The model learns from historical observed temperatures but receives estimated temperatures when forecasting.

In [5]:
# Make a reusable function that converts the small table into model inputs.
def make_features(table):
    features = pd.DataFrame(index=table.index)
    features['temperature_c'] = table['temperature_c']
    features['temperature_squared'] = table['temperature_c'] ** 2
    features['month'] = table['date'].dt.month.astype(str)
    features['weekday'] = table['date'].dt.dayofweek.astype(str)
    return pd.get_dummies(features, columns=['month', 'weekday'], dtype=float)

# Learn the weather assumption from training data only.
monthly_temperature = train.groupby(train['date'].dt.month)['temperature_c'].mean()
forecast_inputs = test[['date']].copy()
forecast_inputs['temperature_c'] = forecast_inputs['date'].dt.month.map(monthly_temperature)

# Ensure training and prediction tables have exactly the same feature columns.
X_train = make_features(train)
y_train = train['demand_mwh']
X_test = make_features(forecast_inputs).reindex(columns=X_train.columns, fill_value=0)
print('Assumed January temperature:', round(monthly_temperature.loc[1], 1), '°C')
display(X_train.head())


Assumed January temperature: 11.1 °C


,temperature_c,temperature_squared,month_1,month_10,month_11,month_12,month_2,month_3,month_4,month_5,...,month_7,month_8,month_9,weekday_0,weekday_1,weekday_2,weekday_3,weekday_4,weekday_5,weekday_6
0,7.400,54.760000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,5.275,27.825625,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,6.375,40.640625,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,8.350,69.722500,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,10.750,115.562500,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## 6. Train one model and a simple baseline

Linear regression learns how the input features relate to demand. `fit()` learns from training data; `predict()` produces estimates for new inputs.

We also need a simple comparison, called a **baseline**. Repeat the final seven days of 2024 across January. All 31 predictions are made from the same December 31 starting point. We do not use January demand to update these forecasts.

In [6]:
# Fit one model without a parameter search.
model = LinearRegression()
model.fit(X_train, y_train)

# Predict January demand, keeping physically impossible negative estimates out.
results = test[['date', 'demand_mwh']].copy()
results['regression_forecast_mwh'] = np.maximum(0, model.predict(X_test))

# Repeat the last available week for a fair, simple comparison.
last_week = train.tail(7)['demand_mwh'].to_numpy()
results['baseline_forecast_mwh'] = np.resize(last_week, len(results))
display(results.head())


,date,demand_mwh,regression_forecast_mwh,baseline_forecast_mwh
2192,2025-01-01,1086807.0,1.012423e+06,1018790.0
2193,2025-01-02,1147154.0,1.012929e+06,1077305.0
2194,2025-01-03,1106968.0,1.008214e+06,1093843.0
2195,2025-01-04,1065018.0,9.725819e+05,1079096.0
2196,2025-01-05,1136098.0,9.628210e+05,1039828.0


## 7. Check the forecasts using MAE

**MAE** means mean absolute error: the average size of the prediction errors, in MWh. Lower is better.

If actual demand is 100 MWh and a prediction is 110 MWh, its absolute error is 10 MWh. We average that kind of error across January. A model does not have to win: an honest explanation of its result is part of data science.

In [7]:
# Compare both methods using the same 31 actual daily values.
model_mae = mean_absolute_error(results['demand_mwh'], results['regression_forecast_mwh'])
baseline_mae = mean_absolute_error(results['demand_mwh'], results['baseline_forecast_mwh'])
scores = pd.DataFrame({'method': ['Linear regression', 'Repeat-week baseline'],
                       'MAE_MWh': [model_mae, baseline_mae]})
display(scores.round(0))
print('Lower MAE:', 'Linear regression' if model_mae < baseline_mae else 'Repeat-week baseline')

# Plot actual and predicted daily demand on the same dates.
results.set_index('date')[['demand_mwh', 'regression_forecast_mwh', 'baseline_forecast_mwh']].div(1_000_000).plot(figsize=(11, 4))
plt.title('January 2025: one forecast made at the end of 2024')
plt.xlabel('Date')
plt.ylabel('Million MWh per day')
plt.tight_layout()
plt.show()


,method,MAE_MWh
0,Linear regression,305335.0
1,Repeat-week baseline,236550.0


Lower MAE: Repeat-week baseline


/var/folders/51/k8dwm_2d6zx1zlrm0dq6jf9w0000gp/T/ipykernel_32679/1348959918.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Produce daily, weekly and monthly outputs

The daily predictions already exist. To obtain weekly or monthly energy, add the daily values.

January starts and ends partway through calendar weeks. We label those partial weeks so nobody mistakes them for seven-day totals. We evaluate only daily MAE here: one month is too little evidence to claim reliable monthly performance.

In [8]:
# Group daily results into Monday–Sunday weeks and a calendar month.
value_columns = ['demand_mwh', 'regression_forecast_mwh', 'baseline_forecast_mwh']
results['week_start'] = results['date'] - pd.to_timedelta(results['date'].dt.dayofweek, unit='D')
weekly = results.groupby('week_start')[value_columns].sum()
weekly['days'] = results.groupby('week_start').size()
weekly['complete_week'] = weekly['days'] == 7
monthly = results.groupby(results['date'].dt.to_period('M'))[value_columns].sum()

# Check that aggregation has not lost or added energy.
assert np.isclose(monthly['regression_forecast_mwh'].sum(), results['regression_forecast_mwh'].sum())
display(weekly.round(0))
display(monthly.round(0))

# Save the results for a simple portfolio write-up.
output = ROOT / 'reports/beginner'
output.mkdir(parents=True, exist_ok=True)
results.to_csv(output / 'daily_forecasts.csv', index=False)
weekly.to_csv(output / 'weekly_forecasts.csv')
monthly.to_csv(output / 'monthly_forecasts.csv')
scores.to_csv(output / 'model_comparison.csv', index=False)
print('Saved files to:', output)


,demand_mwh,regression_forecast_mwh,baseline_forecast_mwh,days,complete_week
week_start,,,,,
2024-12-30,5542045.0,4968968.0,5308862.0,5,False
2025-01-06,10068612.0,6983966.0,7484991.0,7,True
2025-01-13,8948529.0,6983966.0,7484991.0,7,True
2025-01-20,10070241.0,6983966.0,7484991.0,7,True
2025-01-27,5805371.0,5048563.0,5366067.0,5,False


,demand_mwh,regression_forecast_mwh,baseline_forecast_mwh
date,,,
2025-01,40434798.0,30969428.0,33129902.0


Saved files to: <project-root>/reports/beginner


## 9. Explain the outcome in your own words

Use four sentences:

1. **Problem:** “I forecast January electricity demand for ERCOT using calendar features and temperature.”
2. **Method:** “I used SQL to check the data and Python to compare linear regression with repeating the previous week.”
3. **Result:** State the two MAE values printed above and which was lower.
4. **Limitation:** “This is a one-month historical exercise using average expected weather. I would test more months and better weather estimates before relying on it.”

You have now completed a small data science project: question → data checks → preparation → exploration → modeling → evaluation → saved results. Understand this version first; the advanced project can wait.